# Data Heterogeneity Check

This notebook aims to answer whether our three datasets are heterogeneous enough. The main concern is that the Kaggle dataset does not explicitly specify its data source, and therefore it is possible that it comes from Wikidata or OurAirports, which hinders data heterogeneity and makes the data integration less compelling.

Actually, on the [Kaggle Airport Dataset](https://www.kaggle.com/datasets/samvelkoch/global-airports-iata-icao-timezone-geo) page in the provenance section it is mentioned that the data source is "FlightRank 2025: Aeroclub RecSys Cup, 2025". 

I have furthered checked the competition's Kaggle page and found out that airport data is only a piece of the data that the competition provided, with the following structure:
```
{
  "$id": "string",
  "airport": {
    "$id": "string",
    "id": integer,                // Airport ID
    "code": "string",             // Airport code
    "iata": "string",             // IATA code ("SVO", "LED")
    "icao": "string",             // ICAO code ("UUEE", "UWGG")
    "city": {                     // City
      "$id": "string",
      "id": integer,              // City ID
      "code": "string",           // City code
      "iata": "string",           // City IATA code
      "country": {                // Country
        "$id": "string",
        "id": integer,            // Country ID
        "codeA2": "string",       // 2-letter code ("RU", "KZ")
        "codeA3": "string"        // 3-letter code ("RUS", "KAZ")
      }
    }
  },
  "terminal": "string"            // Terminal (optional)
}
```
This still does not align with the `Global Airports Dataset`'s schema (zB: no airport timezone info), so the provenance information remains suspecious.

Therefore, we conduct heterogeneity check for all three pairs of datasets. 

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

df_wikidata = pd.read_csv(project_root / "data" / "interim" / "wikidata_airports_deduped.csv")
df_wikidata = df_wikidata[~df_wikidata['icao'].isna()]
df_wikidata.shape

(11490, 13)

In [2]:
df_ourairports = pd.read_csv(project_root / "data" / "interim" / "ourairports_airports.csv")
df_ourairports = df_ourairports[~df_ourairports['icao_code'].isna()]

df_ourairports.shape

(10530, 19)

In [3]:
df_kaggle = pd.read_csv(project_root / "data" / "interim" / "airports-kaggle.csv")
df_kaggle = df_kaggle[~df_kaggle['ICAO'].isna()]
df_kaggle['ICAO'] = df_kaggle['ICAO'].str.strip().str.upper()

df_kaggle.shape

(5089, 13)

## Pairwise Join and Heterogeneity Check
For each dataset pair, we:
- Inner merge the pair on ICAO code
- Compute the proportion of matched airports with
  - Identical name (after lowercasing and stripping)
  - Identical city (Kaggle–OurAirports only; Wikidata has no city attribute)
  - Identical coordinates (exact copy test)
  - Coordinates within 100 m, and the median distance


### Helper functions

Two small helpers shared by all pairs. Name comparison must normalise first: Kaggle `AirportName` is space-padded for 493 rows (e.g. `'Kapit               '`), OurAirports has 133 names with double spaces.

In [4]:
import numpy as np


def normalize_name(s):
    """Lowercase, strip, and collapse repeated whitespace. Return None for missing values."""
    if pd.isna(s):
        return None
    return " ".join(str(s).lower().split())


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Works element-wise on pandas Series."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

### Align columns to a common schema

Rename each dataset to `icao`, `name`, `city`, `lat`, `lon` so that one comparison function can handle every pair.

| Common | Kaggle | OurAirports | Wikidata |
|---|---|---|---|
| `icao` | `ICAO` | `icao_code` (only; same definition as proposal Table 6, no `ident` fallback) | `icao` |
| `name` | `AirportName` | `name` | `name` |
| `city` | `City_Name` | `municipality` | — (only `located_in`, not a city) |
| `lat` / `lon` | `GeoPointLat` / `GeoPointLong` | `latitude_deg` / `longitude_deg` | `lat` / `lon` |

In [5]:
def align(df, icao, name, lat, lon, city=None):
    """Return a DataFrame with the common columns; `name_raw` keeps the original text for display."""
    out = pd.DataFrame(
        {
            "icao": df[icao].str.strip().str.upper(),
            "name_raw": df[name],
            "name": df[name].map(normalize_name),
            "city": df[city].map(normalize_name) if city else None,
            "lat": df[lat],
            "lon": df[lon],
        }
    )
    return out.drop_duplicates("icao").reset_index(drop=True)


kaggle = align(df_kaggle, "ICAO", "AirportName", "GeoPointLat", "GeoPointLong", city="City_Name")
ourairports = align(df_ourairports, "icao_code", "name", "latitude_deg", "longitude_deg", city="municipality")
wikidata = align(df_wikidata, "icao", "name", "lat", "lon")

for label, d in [("Kaggle", kaggle), ("OurAirports", ourairports), ("Wikidata", wikidata)]:
    assert d["icao"].is_unique
    print(f"{label}: {len(d):,} airports with ICAO")

Kaggle: 5,089 airports with ICAO
OurAirports: 10,530 airports with ICAO
Wikidata: 11,490 airports with ICAO


### Pairwise inner join on ICAO

Three pairs: Kaggle–OurAirports, Kaggle–Wikidata, and **OurAirports–Wikidata as the reference pair** (two sources we already know are independent of Kaggle). Without a reference, a number like "23% identical names" has no baseline.

In [6]:
pairs = {
    "Kaggle vs OurAirports": (kaggle, ourairports, True),
    "Kaggle vs Wikidata": (kaggle, wikidata, False),
    "OurAirports vs Wikidata (reference)": (ourairports, wikidata, False),
}
for label, (left, right, _) in pairs.items():
    print(f"{label}: {len(left.merge(right, on='icao')):,} matched airports")

Kaggle vs OurAirports: 4,700 matched airports
Kaggle vs Wikidata: 4,524 matched airports
OurAirports vs Wikidata (reference): 7,758 matched airports


### Metrics per pair

For every pair, over the matched airports:

| Metric | Reading |
|---|---|
| identical name (normalised) | a copy would be close to 100% |
| identical city | Kaggle–OurAirports only |
| identical coordinates (exact) | copy test; expected to be ~0% everywhere, so not discriminative alone |
| coordinates within 100 m | how close the two sources are even if not copied |
| median distance (km) | robust summary of the coordinate gap |

In [7]:
def compare_pair(left, right, has_city=False):
    """Inner-join `left` and `right` on `icao` and return a dict of the metrics above."""
    m = left.merge(right, on="icao", suffixes=("_l", "_r"))
    dist = haversine_km(m["lat_l"], m["lon_l"], m["lat_r"], m["lon_r"])
    valid = dist.notna()
    return {
        "matched": len(m),
        "name_identical": (m["name_l"] == m["name_r"]).mean(),
        "city_identical": (m["city_l"] == m["city_r"]).mean() if has_city else np.nan,
        "coord_identical": ((m["lat_l"] == m["lat_r"]) & (m["lon_l"] == m["lon_r"])).mean(),
        "within_100m": (dist[valid] <= 0.1).mean(),
        "median_km": dist.median(),
    }


summary = pd.DataFrame({label: compare_pair(l, r, c) for label, (l, r, c) in pairs.items()}).T
summary["matched"] = summary["matched"].astype(int)
summary.round(3)

,matched,name_identical,city_identical,coord_identical,within_100m,median_km
Kaggle vs OurAirports,4700,0.227,0.455,0.01,0.699,0.014
Kaggle vs Wikidata,4524,0.233,NaN,0.00,0.687,0.022
OurAirports vs Wikidata (reference),7758,0.665,NaN,0.00,0.561,0.068


### Look at actual examples

Numbers can hide the *kind* of difference (abbreviations, missing "Airport", different languages). Show names side by side for airports that exist in all three sources.

In [8]:
triple = (
    kaggle[["icao", "name_raw"]].rename(columns={"name_raw": "kaggle"})
    .merge(ourairports[["icao", "name_raw"]].rename(columns={"name_raw": "ourairports"}), on="icao")
    .merge(wikidata[["icao", "name_raw"]].rename(columns={"name_raw": "wikidata"}), on="icao")
)
triple["kaggle"] = triple["kaggle"].str.strip()
print(f"{len(triple):,} airports in all three sources")
triple.sample(15, random_state=0)

4,350 airports in all three sources


,icao,kaggle,ourairports,wikidata
4247,RPVJ,Masbate Airport,Moises R. Espinosa Airport,Moises R. Espinosa Airport
543,PAEG,Eagle Airport,Eagle Airport,Eagle Airport
2953,KLDJ,Linden Airport,Linden Airport,Linden Airport
3694,CYAS,Kangirsuk Airport,Kangirsuk Airport,Kangirsuk Airport
2124,KBRO,Brownsville South Padre Island Intl,Brownsville South Padre Island International A...,Brownsville/South Padre Island International A...
3006,EFKU,Kuopio,Kuopio Airport,Kuopio Airport
3647,DTTX,Thyna,Sfax Thyna International Airport,Sfax–Thyna International Airport
1577,SECU,Mariscal Lamar,Mariscal Lamar Airport,Mariscal Lamar International Airport
402,FMMG,Antsalova Airport,Antsalova Airport,Antsalova Airport
1875,KALO,Waterloo Regional Airport,Waterloo Regional Airport,Waterloo Regional Airport


### Share of abbreviated names (optional)

`Intl` / `Muni` / `Rgnl` / `Regl` / `Fld` / `Arpt` as a share of names per dataset: a style fingerprint that a plain copy would not change.

In [9]:
ABBREV = r"\b(?:Intl|Muni|Rgnl|Regl|Fld|Arpt)\b"
pd.Series(
    {
        "Kaggle": kaggle["name_raw"].str.contains(ABBREV).mean(),
        "OurAirports": ourairports["name_raw"].str.contains(ABBREV).mean(),
        "Wikidata": wikidata["name_raw"].str.contains(ABBREV).mean(),
    },
    name="share of names with an abbreviation",
).round(4)

Kaggle         0.1244
OurAirports    0.0001
Wikidata       0.0001
Name: share of names with an abbreviation, dtype: float64

## Conclusion

| Pair | Matched | Identical name | Identical coordinates | Within 100 m | Median distance |
|---|---|---|---|---|---|
| Kaggle vs OurAirports | 4,700 | 22.7% | 1.0% | 69.9% | 14 m |
| Kaggle vs Wikidata | 4,524 | 23.3% | 0.0% | 68.7% | 22 m |
| OurAirports vs Wikidata (reference) | 7,758 | 66.5% | 0.0% | 56.1% | 68 m |

- **Names**: only about 23% of Kaggle names equal the OurAirports / Wikidata name, against 66.5% between the two reference sources. A copy would be close to 100%.
- **Coordinates**: virtually none are exactly identical (1.0% and 0.0%), so Kaggle is not a copy. The coordinates are nevertheless close (about 70% within 100 m), as expected for the same real-world airports; Kaggle is even closer to the other two than they are to each other, so its coordinates look like a reasonably accurate independent source rather than a degraded one.
- **Naming style**: 12.4% of Kaggle names contain an abbreviation (`Intl`, `Muni`, `Rgnl`, ...), versus 0.01% in OurAirports and Wikidata. Kaggle also often drops "Airport" and uses short local names (`Kuopio`, `Thyna`, `Wang An`). This is the heterogeneity name matching will have to deal with.
- **City**: 45.5% identical for Kaggle vs OurAirports, but this is an exact string comparison; Kaggle writes some cities as `City, State`, so this understates the real agreement.
- **Reading**: Kaggle is not a copy of OurAirports or Wikidata and is heterogeneous enough to keep as the third source.

**Limitations**
- Only airports with an ICAO code are compared. About 20% of Kaggle rows have none, so they cannot be checked this way.
- The origin of Kaggle's name / coordinate / timezone columns is still undocumented; this check only rules out two candidate sources.
- Kaggle's 319 rows at coordinates (0, 0) all have no ICAO, so they never enter this comparison.